# LLM - fine tuning

## dummy code

In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "Qwen/Qwen2-0.5B"

# Load tokenizer & model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,  # use float32 if no GPU
    device_map="auto"
)

messages = [
    {"role": "system", "content": "Tu es un assistant qui répond uniquement en français."},
    {"role": "user", "content": "J'ai besoin de trois idées pour un court de machine learning en français."}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(text, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=150
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


system
Tu es un assistant qui répond uniquement en français.
user
J'ai besoin de trois idées pour un court de machine learning en français.
assistant
Bien sûr, voici trois idées pour un court de machine learning en français :

1. Utiliser des algorithmes de classification pour classer des données en fonction de certaines caractéristiques.
2. Utiliser des algorithmes de prédictions pour prévoir les résultats d'un modèle.
3. Utiliser des algorithmes de recommandation pour fournir des suggestions de produits ou d'informations à partir d'un ensemble de données.

Ces idées sont basées sur les principes de classification, prédictions et recommandation, qui sont des algorithmes couramment utilisés dans le domaine de la machine learning. Les algorithmes de classification peuvent être utilisés pour classer des données en fonction de certaines caractéristiques


## datasets

In [12]:
import os
os.listdir(".")

['llm_example.ipynb', 'celine_pairs.csv']

## fine tuning

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer

tokenizer.pad_token = tokenizer.eos_token

# 2. Manual Tokenization
def tokenize_function(example):
    # Manually creating the ChatML string
    full_text = (f"<|im_start|>system\nTu es Céline.<|im_end|>\n"
                 f"<|im_start|>user\n{example['normal']}<|im_end|>\n"
                 f"<|im_start|>assistant\n{example['celine']}<|im_end|>")
    
    res = tokenizer(full_text, truncation=True, max_length=512, padding="max_length")
    # In Causal LM training, labels are the same as input_ids
    res["labels"] = res["input_ids"].copy()
    return res

dataset = load_dataset("csv", data_files="celine_pairs.csv")
tokenized_dataset = dataset.map(tokenize_function, remove_columns=dataset["train"].column_names)

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

In [ ]:

trainer = Trainer(
    model=model,
    args=TrainingArguments(
        output_dir="./qwen-standard",
        per_device_train_batch_size=4,
        num_train_epochs=3,
        logging_steps=10,
        learning_rate=5e-2,
    ),
    train_dataset=tokenized_dataset["train"],
)

trainer.train()

Step,Training Loss
10,106.797302


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=15, training_loss=71.19820149739583, metrics={'train_runtime': 26.8897, 'train_samples_per_second': 2.231, 'train_steps_per_second': 0.558, 'total_flos': 65967780003840.0, 'train_loss': 71.19820149739583, 'epoch': 3.0})

In [28]:
messages = [
    {"role": "system", "content": "Tu es un assistant qui répond uniquement en français."},
    {"role": "user", "content": "J'ai besoin de trois idées pour un court de machine learning en français."}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(text, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=150
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


system
Tu es un assistant qui répond uniquement en français.
user
J'ai besoin de trois idées pour un court de machine learning en français.
assistant
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!


## start again

In [ ]:
model_name = "Qwen/Qwen2-0.5B"

# Load tokenizer & model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,  # use float32 if no GPU
    device_map="auto"
)


In [2]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForLanguageModeling

# 1. Setup & Model
model_id = "Qwen/Qwen2.5-0.5B"
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16
)


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [3]:

# 2. Manual Label Masking (The "Anti-Bégaiement" Logic)
def tokenize_and_mask(examples):
    prompts = [f"<|im_start|>user\n{s}<|im_end|>\n<|im_start|>assistant\n" for s in examples["normal"]]
    responses = [f"{c}<|im_end|>" for c in examples["celine"]]
    
    inputs = tokenizer(prompts, text_pair=responses, truncation=True, max_length=512, padding="max_length")
    
    # We need to find where the prompt ends to mask it in the labels
    labels = []
    for i, prompt in enumerate(prompts):
        # Tokenize just the prompt to find its length
        prompt_ids = tokenizer(prompt, truncation=True, max_length=512)["input_ids"]
        prompt_len = len(prompt_ids)
        
        # Labels: -100 for the prompt (ignored), actual IDs for the response
        full_ids = inputs["input_ids"][i]
        label = [-100] * prompt_len + full_ids[prompt_len:]
        
        # Ensure label matches input length (handle padding)
        # Anything that is a pad_token in input should be -100 in labels
        label = [l if idx != tokenizer.pad_token_id else -100 for idx, l in zip(full_ids, label)]
        labels.append(label)
        
    inputs["labels"] = labels
    return inputs

# 3. Data Processing
dataset = load_dataset("csv", data_files="celine_pairs.csv", split="train")
tokenized_dataset = dataset.map(tokenize_and_mask, batched=True, remove_columns=dataset.column_names)

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

In [10]:

# 4. Training Arguments
training_args = TrainingArguments(
    output_dir="./celine_qwen_final",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=5e-2,      # Very low to avoid the '!!!!!!' madness
    num_train_epochs=5,       # More epochs since the learning rate is lower
    weight_decay=0.1,
    logging_steps=2,
    save_strategy="epoch",
    bf16=True,
    report_to="none"          # Avoids the 'wandb' error entirely
)

# 5. The Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
)

trainer.train()
trainer.save_model("./celine_qwen_final")

Step,Training Loss
2,21.915770
4,163.023804
6,164.678406
8,161.344376
10,181.784271


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [9]:
model_path = "./celine_qwen_final"

# 1. Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_path)

# 2. Load the model
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.bfloat16, # Match the training precision
    device_map="auto"           # Automatically puts it on GPU if available
)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [11]:
messages = [
    {"role": "system", "content": "Tu es un assistant qui répond uniquement en français."},
    {"role": "user", "content": "J'ai besoin de trois idées pour un court de machine learning en français."}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(text, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=150
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


system
Tu es un assistant qui répond uniquement en français.
user
J'ai besoin de trois idées pour un court de machine learning en français.
assistant
......................................................................................................................................................


## lora

In [10]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# 1. Modèle et Tokenizer
model_id = "Qwen/Qwen2.5-0.5B"
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16
)

# 2. Configuration LoRA
# r=8 ou 16 est suffisant pour un style littéraire
lora_config = LoraConfig(
    r=16, 
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"], # Les couches d'attention à adapter
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# Appliquer LoRA au modèle
model = get_peft_model(model, lora_config)
model.print_trainable_parameters() # Affiche le peu de paramètres qu'on va entraîner

# 3. Préparation des données (Identique au précédent)
def tokenize_and_mask(examples):
    prompts = [f"<|im_start|>user\n{s}<|im_end|>\n<|im_start|>assistant\n" for s in examples["normal"]]
    responses = [f"{c}<|im_end|>" for c in examples["celine"]]
    inputs = tokenizer(prompts, text_pair=responses, truncation=True, max_length=512, padding="max_length")
    
    labels = []
    for i, prompt in enumerate(prompts):
        prompt_len = len(tokenizer(prompt)["input_ids"])
        full_ids = inputs["input_ids"][i]
        label = [-100] * prompt_len + full_ids[prompt_len:]
        label = [l if idx != tokenizer.pad_token_id else -100 for idx, l in zip(full_ids, label)]
        labels.append(label)
    inputs["labels"] = labels
    return inputs

dataset = load_dataset("csv", data_files="celine_pairs.csv", split="train")
tokenized_dataset = dataset.map(tokenize_and_mask, batched=True, remove_columns=dataset.column_names)

# 4. Entraînement
training_args = TrainingArguments(
    output_dir="./qwen-celine-lora",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=1e-3, # Avec LoRA, on peut utiliser un learning rate un peu plus haut
    num_train_epochs=5,
    logging_steps=10,
    save_strategy="epoch",
    bf16=True,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
)

trainer.train()

# 5. Sauvegarde
# Attention : LoRA ne sauvegarde que les "adapters" (quelques Mo)
model.save_pretrained("./qwen-celine-lora-adapter")
tokenizer.save_pretrained("./qwen-celine-lora-adapter")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

trainable params: 1,081,344 || all params: 495,114,112 || trainable%: 0.2184


Step,Training Loss
10,3.696310


('./qwen-celine-lora-adapter/tokenizer_config.json',
 './qwen-celine-lora-adapter/chat_template.jinja',
 './qwen-celine-lora-adapter/tokenizer.json')

In [11]:
messages = [
    {"role": "system", "content": "Tu es un assistant qui répond uniquement en français."},
    {"role": "user", "content": "J'ai besoin de trois idées pour un court de machine learning en français."}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(text, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=150
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


system
Tu es un assistant qui répond uniquement en français.
user
J'ai besoin de trois idées pour un court de machine learning en français.
assistant
Tu as besoin de trois idées pour un court de machine learning en français.ContentLoaded
ContentLoadedassistant
J'ai besoin de trois idées pour un court de machine learning en français.ContentLoadedassistant
ContentLoadedassistant
ContentLoadedassistant
ContentLoadedassistant
ContentLoadedassistant
ContentLoadedassistant
ContentLoadedassistant
ContentLoadedassistant
ContentLoadedassistant
ContentLoadedassistant
ContentLoadedassistant
ContentLoadedassistant
ContentLoadedassistant
ContentLoadedassistant
ContentLoadedassistant
ContentLoadedassistant
ContentLoadedassistant
ContentLoadedassistant
ContentLoadedassistant
ContentLoadedassistant
ContentLoadedassistant
ContentLoadedassistant
ContentLoadedassistant
ContentLoadedassistant
ContentLoadedassistant
ContentLoadedassistant
ContentLoadedassistant
ContentLoadedassistant
ContentLoadedassistant